# 🔄 Customer Retention & Cohort Analysis
Phân tích hành vi giữ chân khách hàng (Retention Rate) theo phân khúc (B2B vs B2C), 
mức thu nhập, và danh mục sản phẩm, sử dụng kỹ thuật Cohort Analysis.

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 0. GLOBAL CONFIGURATION & STYLING
# ==============================================================================
DATA_DIR = "data"
sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "DejaVu Sans"

## 1. Data Ingestion & Master Dataset Construction
**Technical Debt Fix:** Khắc phục lỗi tính sai `FirstDate` bằng cách join toàn bộ dữ liệu 
và tìm ngày mua đầu tiên trên toàn cục (Global Dataset) TRƯỚC KHI lọc các giao dịch theo năm,
đảm bảo tỷ lệ Retention không bị nhiễu bởi khách hàng cũ quay lại.

In [ ]:
# Đọc dữ liệu
df_ecomsales = pd.read_csv(os.path.join(DATA_DIR, "EcomSales.csv"))
df_customers = pd.read_csv(os.path.join(DATA_DIR, "Customer.csv"))
df_products = pd.read_csv(os.path.join(DATA_DIR, "Product.csv"))
df_regions = pd.read_csv(os.path.join(DATA_DIR, "Region.csv"))

# Ép kiểu datetime
df_ecomsales["OrderDate"] = pd.to_datetime(df_ecomsales["OrderDate"])
df_customers["BirthDate"] = pd.to_datetime(df_customers["BirthDate"])

# Lọc dữ liệu hợp lệ (có CustomerID và Quantity > 0)
df_sales = df_ecomsales[(df_ecomsales['CustomerID'].notna()) & (df_ecomsales['Quantity'] > 0)].copy()

# Xây dựng Bảng Master (Tối ưu Runtime, tránh merge lắt nhắt nhiều lần)
df_master = (
    df_sales.merge(df_customers, on="CustomerID", how="left")
    .merge(df_products, on="ProductCode", how="left")
    .merge(df_regions, on="RegionCode", how="left")
)

# TÍNH TOÁN FIRST DATE TOÀN CỤC (GLOBAL)
df_first = df_master.groupby('CustomerID', as_index=False).agg(FirstPurchaseDate=('OrderDate', 'min'))
df_master = df_master.merge(df_first, on='CustomerID', how='inner')

# Tính toán các cột chuẩn bị cho Cohort
df_master['AcquisitionMonth'] = df_master['FirstPurchaseDate'].dt.to_period('M')
df_master['MonthAfter'] = (df_master['OrderDate'].dt.year - df_master['FirstPurchaseDate'].dt.year) * 12 + \
                          (df_master['OrderDate'].dt.month - df_master['FirstPurchaseDate'].dt.month)

# Xử lý Income Group (Điền NaNs bằng Median và chia 3 nhóm)
median_income = df_master['AnnualIncome'].median()
df_master['AnnualIncome'] = df_master['AnnualIncome'].fillna(median_income)
df_master['IncomeGroup'] = pd.qcut(df_master['AnnualIncome'], q=3, labels=['Low Income', 'Medium Income', 'High Income'])

def build_cohort_matrix(df, value_col='CustomerID', agg_func='nunique'):
    """Hàm tạo ma trận Retention (Tỷ lệ) và Cohort Sizes (Số lượng tuyệt đối)"""
    cohort_data = df.pivot_table(index='AcquisitionMonth', columns='MonthAfter', 
                                 values=value_col, aggfunc=agg_func)
    # Cột 0 chính là số lượng khách hàng mới của tháng đó
    cohort_sizes = cohort_data.iloc[:, 0]
    retention_matrix = cohort_data.divide(cohort_sizes, axis=0)
    return cohort_data, retention_matrix, cohort_sizes

## 2. Khám phá Cohort Khách hàng mới năm 2022
**Update:** Bổ sung biểu đồ Bar Chart hiển thị Acquisition Volume (Số lượng khách mới) 
đặt cạnh Heatmap theo đúng chuẩn phân tích Cohort quốc tế.

In [ ]:
# Lọc những khách hàng THỰC SỰ GIA NHẬP trong năm 2022
df_cohort_2022 = df_master[df_master['FirstPurchaseDate'].dt.year == 2022].copy()
_, ret_2022, sizes_2022 = build_cohort_matrix(df_cohort_2022)

# Thiết lập layout 1x2 (Cột trái cho Volume, cột phải cho Heatmap)
fig, (ax_bar, ax_heat) = plt.subplots(
    1, 2, figsize=(18, 8), dpi=100, gridspec_kw={'width_ratios': [1, 4]}
)

# 1. Bar Chart: Acquisition Volume
y_pos = np.arange(len(sizes_2022.index))
ax_bar.barh(y_pos, sizes_2022.values, color='#3498db', edgecolor='none')
ax_bar.set_yticks(y_pos)
ax_bar.set_yticklabels(sizes_2022.index.astype(str))
ax_bar.invert_yaxis()  # Đảo ngược trục Y để khớp với Heatmap
ax_bar.set_title('Acquisition Volume\n(New Customers)', fontsize=12, fontweight='bold')
ax_bar.set_xlabel('Users')
for i, v in enumerate(sizes_2022.values):
    ax_bar.text(v + 5, i, f'{v:,.0f}', va='center', fontsize=10)

# 2. Heatmap: Retention Rate
sns.heatmap(ret_2022, annot=True, fmt='.0%', cmap='YlGnBu', vmin=0.0, vmax=0.15, ax=ax_heat)
ax_heat.set_title('Retention Rate: 2022 Cohort', fontsize=16, fontweight='bold', pad=15)
ax_heat.set_ylabel('') # Đã có nhãn bên Bar chart
ax_heat.set_xlabel('Tháng sau mua (Month After)', fontsize=11)
ax_heat.set_yticks([]) # Ẩn trục Y vì đã đồng bộ với Bar Chart

plt.tight_layout()
plt.show()

### 💡 Insight: Bức Tranh Toàn Cảnh 2022
* **Acquisition Volume:** Quý 1 thu hút lượng khách hàng mới rất tốt, nhưng có xu hướng giảm dần về cuối năm.
* **Churn Rate Nghiêm Trọng:** Tỷ lệ rớt đài ngay tháng đầu tiên cực cao (Retention dao động 1% - 3%). Hệ thống đang thiếu một cơ chế on-boarding hoặc email nhắc nhở mua lại trong 30 ngày đầu.

## 3. So Sánh Retention Theo Segment: Corporate vs. Consumer

In [ ]:
df_corp = df_cohort_2022[df_cohort_2022['Segment'] == 'Corporate'].copy()
df_cons = df_cohort_2022[df_cohort_2022['Segment'] == 'Consumer'].copy()

_, ret_corp, _ = build_cohort_matrix(df_corp)
_, ret_cons, _ = build_cohort_matrix(df_cons)

fig, axes = plt.subplots(1, 2, figsize=(22, 8), dpi=100, sharey=True)

sns.heatmap(ret_corp, annot=True, fmt='.0%', cmap='Blues', ax=axes[0], vmin=0, vmax=0.15)
axes[0].set_title('Retention Rate: CORPORATE (B2B)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Tháng sau mua')
axes[0].set_ylabel('Acquisition Month')

sns.heatmap(ret_cons, annot=True, fmt='.0%', cmap='Greens', ax=axes[1], vmin=0, vmax=0.15)
axes[1].set_title('Retention Rate: CONSUMER (B2C)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Tháng sau mua')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

### 💡 Insight: B2B vs B2C
* **Corporate (B2B):** Có những "điểm sáng" chu kỳ (tháng thứ 5, thứ 8 với Retention đạt 5%-6%). Họ mua rải rác nhưng mang tính chất restock định kỳ. -> *Đề xuất: Tạo luồng CRM tự động cho B2B Sales gọi điện chăm sóc vào chu kỳ 4-5 tháng.*
* **Consumer (B2C):** Mua ngẫu nhiên, biểu đồ nhạt, tỷ lệ giữ chân rất thấp.

## 4. Phân Tích Lợi Nhuận Khách Hàng Cũ Theo Thu Nhập (Scale vs. Efficiency)
Sử dụng toàn bộ lịch sử giao dịch để xem giá trị mà khách hàng quay lại mang lại.

In [ ]:
profit_sum = df_master[df_master['MonthAfter'] > 0].pivot_table(
    index='IncomeGroup', columns='MonthAfter', values='Profit', aggfunc='sum'
).fillna(0)

# Chuyển đổi Mean Profit thành Average Order Value (AOV / Efficiency)
profit_mean = df_master[df_master['MonthAfter'] > 0].pivot_table(
    index='IncomeGroup', columns='MonthAfter', values='Profit', aggfunc='mean'
).fillna(0)

fig, axes = plt.subplots(2, 1, figsize=(16, 11), dpi=100, sharex=True)

sns.heatmap(profit_sum.iloc[:, :12], annot=True, fmt=',.0f', cmap='Oranges', ax=axes[0]) 
axes[0].set_title('VOLUME (SUM PROFIT) - Nhóm nào mang lại dòng tiền lớn nhất?', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Nhóm Thu Nhập')
axes[0].set_xlabel('') 

sns.heatmap(profit_mean.iloc[:, :12], annot=True, fmt=',.0f', cmap='Greens', ax=axes[1]) 
axes[1].set_title('EFFICIENCY (MEAN PROFIT) - Nhóm nào có tỷ suất sinh lời/đơn tốt nhất?', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Nhóm Thu Nhập')
axes[1].set_xlabel('Tháng quay lại mua sắm (Month 1 -> Month 12)')

plt.tight_layout()
plt.show()

### 💡 Insight: Scale vs Efficiency Paradox
* **High Income (Cash Grab):** Sinh lời rất cao trên mỗi đơn quay lại (biểu đồ xanh lá đậm), nhưng số lượng ít và lười quay lại (Nhiều khoảng trống). -> *Chiến lược: Upsell/Cross-sell mạnh tay ngay trong lần mua đầu tiên.*
* **Low Income (Cash Cow):** Lợi nhuận từng đơn thấp, nhưng mua đông và bền bỉ, tạo ra dòng tiền tổng (Sum Profit) khổng lồ và ổn định. -> *Chiến lược: Đẩy mạnh chương trình Loyalty, tích điểm đổi quà.*

## 5. Phân Tích Cohort Dựa Trên Danh Mục Mua Lần Đầu (First Purchase Category)

In [ ]:
first_cat_df = df_master.sort_values('OrderDate').groupby('CustomerID').first().reset_index()
first_cat_mapping = first_cat_df[['CustomerID', 'Category']].rename(columns={'Category': 'First_Category'})

df_cohort_cat = df_cohort_2022.merge(first_cat_mapping, on='CustomerID', how='inner')
top_categories = df_cohort_cat['First_Category'].value_counts().nlargest(2).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(22, 8), dpi=100, sharey=True)

for idx, cat in enumerate(top_categories):
    df_cat = df_cohort_cat[df_cohort_cat['First_Category'] == cat].copy()
    _, ret_cat, _ = build_cohort_matrix(df_cat)
    
    sns.heatmap(ret_cat, annot=True, fmt='.0%', cmap='Reds', ax=axes[idx], vmin=0, vmax=0.15)
    axes[idx].set_title(f'Retention (First Buy: {cat})', fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('Tháng sau mua')

axes[0].set_ylabel('Acquisition Month')
plt.tight_layout()
plt.show()